![image info](https://raw.githubusercontent.com/albahnsen/MIAD_ML_and_NLP/main/images/banner_1.png)

# Creación de un Lambda Layer con scikit-learn

En este notebook se documenta cómo crear un **AWS Lambda Layer** que contenga las librerías necesarias para desplegar modelos de `scikit-learn` en AWS Lambda.

## ¿Por qué necesitamos un Layer?

El tamaño máximo del paquete de despliegue de una función Lambda (`.zip` subido directamente) es de **50 MB comprimido / 250 MB descomprimido**. Las librerías como `scikit-learn`, `numpy` y `scipy` suelen superar este límite cuando se incluyen junto con el código.

Un **Lambda Layer** permite separar las dependencias del código, evitando tener que subirlas en cada despliegue. Además, un mismo layer puede ser compartido entre múltiples funciones Lambda y entre cuentas de AWS.

## Requisitos previos

- Tener instalado el **AWS CLI** y estar autenticado (`aws configure`).
- Tener `pip` y `zip` disponibles en el entorno local.
- Permisos de IAM para crear buckets de S3 y publicar Lambda Layers.

## Paso 1 · Instalar las librerías con la plataforma correcta

AWS Lambda usa Amazon Linux 2 (`manylinux2014_x86_64`). Al instalar las librerías debemos pedirle a `pip` que descargue los **wheels binarios compatibles con esa plataforma**, no los de macOS / Windows. Si no se hace esto, la Lambda fallará al importar `numpy` o `scipy` con errores de arquitectura incompatible.

Las banderas clave son:

| Flag | Significado |
|---|---|
| `--platform manylinux2014_x86_64` | Plataforma objetivo (Amazon Linux 2 x86_64) |
| `--python-version 3.12` | Versión de Python del runtime de Lambda |
| `--implementation cp` | CPython |
| `--only-binary=:all:` | Obliga a usar wheels (no compilar desde fuente) |
| `--target=python` | Las librerías deben ir en un directorio llamado `python/` — es la convención de Lambda Layers |

> ⚠️ **Importante:** la estructura del zip DEBE ser `python/<librerías>` para que Lambda las encuentre en `/opt/python/` en tiempo de ejecución.

In [ ]:
%%bash
rm -rf python sklearn-layer.zip
mkdir -p python

pip install \
  --platform manylinux2014_x86_64 \
  --target=python \
  --implementation cp \
  --python-version 3.12 \
  --only-binary=:all: \
  --upgrade \
  scikit-learn joblib numpy scipy

# Limpiezas SEGURAS para reducir tamaño
find python -type d -name "__pycache__" -exec rm -rf {} + 2>/dev/null
find python -name "*.pyc" -delete
find python -type d -name "*.dist-info" -exec rm -rf {} +

# Borrar tests/data NO esenciales (evitar numpy._core.tests — sí es esencial)
rm -rf python/scipy/io/tests 2>/dev/null
rm -rf python/sklearn/tests 2>/dev/null
rm -rf python/sklearn/datasets/data 2>/dev/null
rm -rf python/sklearn/datasets/images 2>/dev/null
rm -rf python/scipy/misc 2>/dev/null

echo "=== Tamaño descomprimido ==="
du -sh python/

zip -r9 sklearn-layer.zip python -q
echo "=== Tamaño zip ==="
ls -lh sklearn-layer.zip

## Paso 2 · Subir el zip a S3

Los Lambda Layers que superan los 50 MB (como este) no pueden subirse directamente desde la CLI. Deben primero cargarse a un bucket de S3.

> ⚠️ Los nombres de bucket en S3 son **globales**. Cambia `uniandes-ml-nlp-layers` por un nombre único si el tuyo ya existe.

In [ ]:
%%bash
# Crear bucket (solo una vez; falla silenciosamente si ya existe)
aws s3 mb s3://uniandes-ml-nlp-layers --region us-east-1

# Subir el zip
aws s3 cp sklearn-layer.zip s3://uniandes-ml-nlp-layers/sklearn-layer.zip

## Paso 3 · Publicar el Lambda Layer

Cada vez que se publica un layer se crea una **nueva versión** (el número aumenta en 1). Las versiones anteriores siguen existiendo — así las funciones que las usan no se rompen cuando actualizas.

In [ ]:
%%bash
aws lambda publish-layer-version \
  --layer-name sklearn-py312 \
  --description "sklearn 1.7.2 + joblib 1.5.3 + numpy 2.2.6 + scipy 1.16.3" \
  --content S3Bucket=uniandes-ml-nlp-layers,S3Key=sklearn-layer.zip \
  --compatible-runtimes python3.12 \
  --compatible-architectures x86_64 \
  --region us-east-1

La respuesta del comando incluye el **ARN** del layer con su número de versión. Guárdalo — lo necesitarás para adjuntarlo a una función Lambda.

```
arn:aws:lambda:us-east-1:915485756489:layer:sklearn-py312:3
```

## Paso 4 · Hacer el layer público (opcional)

Por defecto, un Lambda Layer solo es accesible desde la cuenta de AWS donde fue creado. Para que los estudiantes puedan adjuntarlo a sus propias Lambdas sin tener que recrearlo, se puede dar permiso público de lectura.

> ⚠️ Cambia `--version-number 3` por la versión que devolvió el paso anterior.

In [ ]:
%%bash
aws lambda add-layer-version-permission \
  --layer-name sklearn-py312 \
  --version-number 3 \
  --statement-id public-access \
  --action lambda:GetLayerVersion \
  --principal "*" \
  --region us-east-1

## Paso 5 · Adjuntar el layer a una función Lambda

Desde la **consola de AWS**:
1. Abre la función Lambda.
2. En la pestaña **Code**, baja hasta la sección **Layers**.
3. Click en **Add a layer** → **Specify an ARN** y pega el ARN del layer.

Desde el **AWS CLI**:

In [ ]:
%%bash
aws lambda update-function-configuration \
  --function-name MI_FUNCION \
  --layers arn:aws:lambda:us-east-1:915485756489:layer:sklearn-py312:3 \
  --region us-east-1

## ARN público del layer del curso

Los estudiantes pueden usar directamente el siguiente ARN en sus funciones Lambda, sin tener que crear el layer desde cero:

```
arn:aws:lambda:us-east-1:915485756489:layer:sklearn-py312:3
```

**Versiones incluidas:**

| Librería | Versión |
|---|---|
| scikit-learn | 1.7.2 |
| joblib | 1.5.3 |
| numpy | 2.2.6 |
| scipy | 1.16.3 |
| threadpoolctl | 3.6.0 |

> ⚠️ **Importante:** al entrenar y serializar un modelo con `joblib.dump()` localmente, usa **exactamente estas mismas versiones** y **Python 3.12**, o el `.pkl` fallará al cargarse en Lambda con errores tipo `KeyError: 239`, `InconsistentVersionWarning` o incompatibilidades binarias entre `numpy` y `scipy`.